In [13]:
from db_connect import get_db_connection

In [31]:
import os
import sys

# Current notebook folder
project_path = os.getcwd()

print(project_path)

# Add project folder to Python path
sys.path.append(project_path)

c:\YR3  SEM2\PRMS SYSTEM\routes


In [30]:
import os
import sys

project_path = os.path.abspath("..")
sys.path.append(project_path)

In [29]:
from db_connect import get_db_connection

In [34]:
conn = get_db_connection()

if conn is None:
    raise Exception("Database connection failed.")

print("Connection successful!")

✅ Database connected successfully
Connection successful!


In [18]:
import sys
print(sys.executable)

c:\YR3  SEM2\PRMS SYSTEM\venv\Scripts\python.exe


In [27]:
import os
print(os.getcwd())

c:\YR3  SEM2\PRMS SYSTEM\routes


In [26]:
import pandas as pd

In [35]:

query = """
SELECT

    -- =========================
    -- COMPANY BUDGET FEATURES
    -- =========================
    cb.company_budget_id,
    cb.financial_year,
    cb.allocated_amount AS company_allocated_amount,
    cb.budget_type AS company_budget_type,


    -- =========================
    -- DEPARTMENT BUDGET FEATURES
    -- =========================
    db.department_budget_id,
    db.department_id,
    db.budget_item,
    db.quantity AS budget_quantity,
    db.unit_cost AS budget_unit_cost,
    db.total_budget AS department_budget,


    -- =========================
    -- REQUISITION FEATURES
    -- =========================
    COUNT(DISTINCT r.requisition_id) AS total_requisitions,

    COUNT(DISTINCT ri.item_id) AS total_requested_items,

    COALESCE(
        SUM(ri.qty_required),
        0
    ) AS requested_quantity,


    -- =========================
    -- PURCHASE ORDER FEATURES
    -- =========================
    COUNT(DISTINCT po.purchase_order_id)
        AS total_purchase_orders,

    COALESCE(
        SUM(poi.quantity),
        0
    ) AS ordered_quantity,


    COALESCE(
        SUM(poi.total_price),
        0
    ) AS purchase_order_cost,


    COALESCE(
        AVG(poi.unit_price),
        0
    ) AS average_unit_price,


    -- =========================
    -- GRN FEATURES
    -- =========================
    COUNT(DISTINCT g.grn_id)
        AS total_grn_received,


    COALESCE(
        SUM(gi.quantity_received),
        0
    ) AS received_quantity,


    COALESCE(
        SUM(gi.quantity_rejected),
        0
    ) AS rejected_quantity,


    -- =========================
    -- AI ENGINEERED FEATURES
    -- =========================

    (
        COALESCE(SUM(poi.total_price),0)
        /
        NULLIF(db.total_budget,0)
    ) * 100 AS budget_utilization_percentage,


    (
        db.total_budget -
        COALESCE(SUM(poi.total_price),0)
    ) AS remaining_budget,


    (
        COALESCE(SUM(gi.quantity_received),0)
        /
        NULLIF(SUM(poi.quantity),0)
    ) * 100 AS delivery_completion_percentage


FROM company_budget cb


INNER JOIN department_budget db
ON cb.company_budget_id = db.company_budget_id


LEFT JOIN requisitions r
ON db.department_id = r.department_id


LEFT JOIN requisition_items ri
ON r.requisition_id = ri.requisition_id


LEFT JOIN purchase_orders po
ON r.requisition_id = po.requisition_id


LEFT JOIN purchase_order_items poi
ON po.purchase_order_id = poi.purchase_order_id


LEFT JOIN grn g
ON po.purchase_order_id = g.po_id


LEFT JOIN grn_items gi
ON g.grn_id = gi.grn_id


GROUP BY

    cb.company_budget_id,
    cb.financial_year,
    cb.allocated_amount,
    cb.budget_type,

    db.department_budget_id,
    db.department_id,
    db.budget_item,
    db.quantity,
    db.unit_cost,
    db.total_budget;
"""

df = pd.read_sql(query, conn)

conn.close()

print(df.head())
print(df.shape)

   company_budget_id  financial_year  company_allocated_amount  \
0                  1            2026                12000000.0   

  company_budget_type  department_budget_id  department_id budget_item  \
0              Yearly                     1              1    computer   

   budget_quantity  budget_unit_cost  department_budget  ...  \
0               12          700000.0          8400000.0  ...   

   total_purchase_orders  ordered_quantity  purchase_order_cost  \
0                      0               0.0                  0.0   

   average_unit_price  total_grn_received  received_quantity  \
0                 0.0                   0                0.0   

   rejected_quantity  budget_utilization_percentage  remaining_budget  \
0                0.0                            0.0         8400000.0   

   delivery_completion_percentage  
0                            None  

[1 rows x 23 columns]
(1, 23)


C:\Users\wawir\AppData\Local\Temp\ipykernel_7832\274129747.py:150: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [24]:

import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

import matplotlib.pyplot as plt
import seaborn as sns